# Step 3 — Cluster Failure Types
**Roadmap reference:** K-Means unsupervised clustering

## Goal
Identify distinct failure archetypes — not just reassignment count.
This tells you *which type* of problem you are solving before you design any fix.

## Expected clusters (from roadmap)
1. **Legitimate complexity escalation** — linear escalation, no bounce, high loss amount
2. **Structural bounce** — same-tier handoffs, low escalation, User-3 pattern
3. **Geographic mismatch routing** — high geo mismatch, system-triggered
4. **Manual referral re-entry** — high manual trigger, referee returns to earlier adjuster

## Method
- StandardScaler → KMeans k=4
- t-SNE (2D) for visualization
- Silhouette score to validate separation

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from pathlib import Path

DATA = Path("data")
matrix = pd.read_csv(DATA / "step2_feature_matrix.csv")

# Use only non-FTR claims for clustering (Group A/B/C — the failure cases)
df_fail = matrix[matrix["final_group"] != "0"].copy().reset_index(drop=True)
print(f"Clustering {len(df_fail)} non-FTR claims")

Clustering 154 non-FTR claims


In [2]:
# Features for clustering
CLUSTER_FEATURES = [
    "assignment_count", "pct_manual", "pct_system", "pct_user_auto",
    "n_unique_adjusters", "same_adjuster_returned", "avg_hrs_between_events",
    "geographic_mismatch", "exposure_added_within_48h",
    "reported_loss_amount",
]

X = df_fail[CLUSTER_FEATURES].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Find optimal k using silhouette score
sil_scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil_scores[k] = round(silhouette_score(X_scaled, labels), 4)

print("Silhouette scores by k:")
for k, s in sil_scores.items():
    print(f"  k={k}: {s}")
best_k = max(sil_scores, key=sil_scores.get)
print(f"\nBest k = {best_k}")

Silhouette scores by k:
  k=2: 0.2033
  k=3: 0.1374
  k=4: 0.1584
  k=5: 0.159
  k=6: 0.16

Best k = 2


In [3]:
# Final clustering
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df_fail["cluster"] = km.fit_predict(X_scaled)

# Label clusters by their dominant characteristics — only average numeric features
cluster_profiles = df_fail.groupby("cluster")[CLUSTER_FEATURES].mean().round(3)
print("Cluster profiles (mean feature values):")
print(cluster_profiles[["assignment_count", "pct_manual", "geographic_mismatch",
                          "same_adjuster_returned", "avg_hrs_between_events"]].to_string())

Cluster profiles (mean feature values):
         assignment_count  pct_manual  geographic_mismatch  same_adjuster_returned  avg_hrs_between_events
cluster                                                                                                   
0                   7.615       0.695                0.000                   0.615                  92.408
1                   4.896       0.735                1.000                   0.375                  96.408
2                   3.517       0.337                0.966                   0.207                  89.441
3                   9.906       0.705                1.000                   0.719                  88.412


In [4]:
# Business labels based on profile inspection
CLUSTER_LABELS = {
    0: "Complexity Escalation",
    1: "Structural Bounce",
    2: "Geographic Mismatch",
    3: "Manual Re-entry",
}
# Auto-assign by highest distinguishing feature
for cid, row in cluster_profiles.iterrows():
    if row["geographic_mismatch"] == cluster_profiles["geographic_mismatch"].max():
        CLUSTER_LABELS[cid] = "Geographic Mismatch"
    elif row["same_adjuster_returned"] == cluster_profiles["same_adjuster_returned"].max():
        CLUSTER_LABELS[cid] = "Structural Bounce (User-3 Pattern)"
    elif row["pct_manual"] == cluster_profiles["pct_manual"].max():
        CLUSTER_LABELS[cid] = "Manual Re-entry / Override"
    else:
        CLUSTER_LABELS[cid] = "Complexity Escalation"

df_fail["cluster_label"] = df_fail["cluster"].map(CLUSTER_LABELS)
print("\nCluster labels assigned:")
print(df_fail["cluster_label"].value_counts().to_string())


Cluster labels assigned:
cluster_label
Geographic Mismatch      112
Complexity Escalation     42


In [5]:
# t-SNE visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
coords = tsne.fit_transform(X_scaled)
df_fail["tsne_x"] = coords[:, 0]
df_fail["tsne_y"] = coords[:, 1]

fig = px.scatter(df_fail, x="tsne_x", y="tsne_y",
                 color="cluster_label",
                 hover_data=["claim_id", "final_group", "assignment_count"],
                 title="Failure Archetype Clusters — t-SNE (Commercial Property 2024)",
                 labels={"tsne_x": "t-SNE 1", "tsne_y": "t-SNE 2",
                         "cluster_label": "Failure Type"})
fig.update_traces(marker=dict(size=6, opacity=0.8))
fig.show()

In [6]:
# Save
df_fail[["claim_id", "cluster", "cluster_label"]].to_csv(
    DATA / "step3_clusters.csv", index=False)
print("Saved → data/step3_clusters.csv")

# Summary table
summary = (df_fail.groupby("cluster_label")
           .agg(n_claims=("claim_id", "count"),
                avg_assignments=("assignment_count", "mean"),
                pct_returned=("same_adjuster_returned", "mean"),
                pct_geo_mismatch=("geographic_mismatch", "mean"))
           .round(2).reset_index())
print("\nCluster summary:")
print(summary.to_string(index=False))

Saved → data/step3_clusters.csv

Cluster summary:
        cluster_label  n_claims  avg_assignments  pct_returned  pct_geo_mismatch
Complexity Escalation        42             4.79          0.33              0.67
  Geographic Mismatch       112             7.76          0.57              1.00
